# compilacao-montar.ipynb — versículos sortidos da Bíblia inteira

> ⚡ **Ligue a GPU** — este notebook roda Whisper. *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU.* Sem ela funciona, mas leva muito mais tempo.

Você escolhe os versículos; o notebook faz o resto. A seleção pode pular entre
livros, repetir capítulo e sair de ordem — o que manda é a ordem que você
escreveu.

## O fluxo

| # | Quem | O quê |
|---|---|---|
| 1 | **você** | escreve o tema e a seleção na Configuração |
| 2 | notebook | extrai o texto dos versículos do `web-biblia.json` |
| 3 | notebook | diz **quais capítulos ainda precisam de tempo** |
| 4 | notebook | transcreve com Whisper e alinha **só os que faltam** |
| 5 | notebook | corta, concatena, gera `.wav` + `.srt` + manifesto |

O passo 4 é o caro, e é o que o cache existe pra você pagar **uma vez por
capítulo, pra sempre**. A segunda compilação que usar Salmo 23 pula ele.

## Pré-requisitos

- `biblia-audio-baixar.ipynb` rodado (os mp3 em `assets/biblia_audio/`)
- `biblia-texto-baixar.ipynb` rodado (`dados_lexico/web-biblia.json`)

## Saída

Em `videos/comp_<tema>/`: o áudio compilado, o SRT recalculado do zero, e o
**manifesto** — de qual versículo original veio cada trecho. É o manifesto que
a montagem de vídeo usa depois pra achar a mídia certa de cada pedaço.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP                                                         ║
# ╚══════════════════════════════════════════════════════════════════╝

!apt-get -qq -y install ffmpeg > /dev/null 2>&1
!pip install -q openai-whisper
print('✅ ffmpeg, whisper')

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

import shutil, sys
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO_MODULOS = Path("/content/pipeline")

if DESTINO_MODULOS.exists():
    shutil.rmtree(DESTINO_MODULOS)
shutil.copytree(PASTA_MODULOS, DESTINO_MODULOS)
if str(DESTINO_MODULOS) not in sys.path:
    sys.path.insert(0, str(DESTINO_MODULOS))
print(f"✅ {len(list(DESTINO_MODULOS.glob('*.py')))} módulos")

# ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
# "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
# traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
# então um copytree logo depois do mount às vezes enxerga só parte dos
# arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
# quebrar muito depois, num import, longe da causa.
#
# A conferência é de três pontas, porque a causa muda o conserto:
#   manifesto  o que o repositório tem  (versionado; chega pela cópia)
#   Drive      o que chegou lá
#   VM         o que a cópia desta célula trouxe
_manifesto = PASTA_MODULOS / "_manifesto.txt"
if not _manifesto.exists():
    print("   ⚠️  sem _manifesto.txt no Drive — ele é versionado no repositório,")
    print("      então rode o repositorio-sincronizar pra trazê-lo")
else:
    _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                  if l.strip() and not l.startswith("#")}
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO_MODULOS.glob("*.py")}

    _fora_do_drive = sorted(_esperados - _no_drive)
    _nao_copiados  = sorted((_esperados & _no_drive) - _na_vm)

    if _nao_copiados:
        # Estão no Drive mas não vieram: é a listagem preguiçosa do mount.
        # Uma segunda passada, com o mount já quente, costuma resolver.
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO_MODULOS / _n)
        _na_vm = {f.name for f in DESTINO_MODULOS.glob("*.py")}
        _nao_copiados = sorted((_esperados & _no_drive) - _na_vm)

    if _fora_do_drive:
        print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
        for _n in _fora_do_drive:
            print(f"     {_n}")
        raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_esperados)} módulos do manifesto estão na VM")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO — edite só esta célula                          ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. TEMA ─────────────────────────────────────────────────────────────────
# Livre, seu. Vira o nome da pasta: "Salmos Esperança" -> comp_salmos_esperanca
# Acento, espaço e pontuação são normalizados; reescrever com outra pontuação
# dá o mesmo nome, então não cria pasta duplicada por engano.
TEMA = "Salmos Esperança"

# ── 2. SELEÇÃO ──────────────────────────────────────────────────────────────
# (sigla do livro, capítulo, versículos)
#   "1-3"      intervalo
#   "5"        um só
#   "1-3,6,9"  mistura
#   [1, 2, 5]  lista, se preferir
#
# A ORDEM AQUI É A ORDEM DO VÍDEO. Pode pular entre livros, repetir capítulo e
# sair de ordem à vontade.
SELECAO = [
    ("Ps",  23, "1-6"),
    ("Ps",  42, "5,11"),
    ("Isa", 40, "28-31"),
    ("Rom",  8, "38-39"),
]

# ── 3. WHISPER ──────────────────────────────────────────────────────────────
# Só usado nos capítulos que ainda não têm tempo. Trocar o modelo invalida o
# cache (tempo calculado com outro modelo é outro tempo), então mude com
# critério -- "base" costuma bastar pro alinhamento, que é por sequência de
# palavras e tolera erro de transcrição.
MODELO_WHISPER = "base"
IDIOMA_AUDIO = "en"        # a narração do David Williams é em inglês

print(f"Tema ...... {TEMA}")
print(f"Seleção ... {len(SELECAO)} entrada(s)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INICIALIZAR                                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

import json, sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

import biblia_livros as bl
import biblia_texto as bt
import tempos_cache as tc
from compilacao_pipeline import (nome_compilacao, conflita_com_capitulo,
                                 parsear_selecao, capitulos_da_selecao,
                                 compilar_selecao)

BASE = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}")
PASTA_AUDIO = BASE / "assets" / "biblia_audio"
PASTA_TEMPOS = BASE / "assets" / "biblia_tempos"
CAMINHO_BIBLIA = BASE / "pipeline" / "dados_lexico" / "web-biblia.json"

for caminho, dica in [(PASTA_AUDIO, "rode biblia-audio-baixar.ipynb"),
                      (CAMINHO_BIBLIA, "rode biblia-texto-baixar.ipynb")]:
    if not caminho.exists():
        raise SystemExit(f"❌ Falta {caminho.name} — {dica}")

NOME = nome_compilacao(TEMA)
if conflita_com_capitulo(NOME):
    raise SystemExit(f"❌ {NOME!r} colide com o nome de um capítulo — troque o tema.")

PASTA_SAIDA = BASE / "videos" / NOME
ITENS = parsear_selecao(SELECAO)

BIBLIA = json.loads(CAMINHO_BIBLIA.read_text(encoding="utf-8"))["livros"]

def versiculos_do_capitulo(livro, capitulo):
    dados = BIBLIA[livro.sigla][str(capitulo)]
    return [bt.Versiculo(v["n"], v["t"], v["q"]) for v in dados]

def texto_de(livro, capitulo, versiculo):
    for v in versiculos_do_capitulo(livro, capitulo):
        if v.numero == versiculo:
            return v.texto
    raise KeyError(f"{livro.sigla} {capitulo}:{versiculo} não está no web-biblia.json")

total = sum(len(vs) for _, _, vs in ITENS)
print(f"📛 {NOME}")
print(f"📁 {PASTA_SAIDA}")
print(f"📖 {total} versículos, {len(capitulos_da_selecao(ITENS))} capítulo(s)")
for livro, cap, vs in ITENS:
    print(f"   {livro.nome} {cap}:{','.join(str(v) for v in vs)}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 1/3 — QUAIS CAPÍTULOS PRECISAM DE TEMPO                       ║
# ╚══════════════════════════════════════════════════════════════════╝

# Confere ANTES de transcrever qualquer coisa: melhor saber a conta inteira
# agora do que descobrir capítulo faltante no meio do processo.
necessarios = capitulos_da_selecao(ITENS)
falta_tempo, falta_audio, prontos = [], [], []

for nome_cap in necessarios:
    if not (PASTA_AUDIO / f"{nome_cap}.mp3").exists():
        falta_audio.append(nome_cap)
        continue
    guardado, motivo = tc.carregar(PASTA_TEMPOS, nome_cap)
    (prontos if guardado is not None else falta_tempo).append((nome_cap, motivo))

print(f"✅ com tempo pronto ... {len(prontos)}")
for nome_cap, _ in prontos:
    print(f"     {nome_cap}")

print(f"\n⏳ precisam transcrever . {len(falta_tempo)}")
for nome_cap, motivo in falta_tempo:
    print(f"     {nome_cap} — {motivo}")

if falta_audio:
    print(f"\n❌ SEM ÁUDIO NO DRIVE ... {len(falta_audio)}")
    for nome_cap in falta_audio:
        print(f"     {nome_cap}.mp3")
    raise SystemExit("Rode o biblia-audio-baixar.ipynb — sem o áudio não dá pra cortar.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎙️ 2/3 — CALCULAR OS TEMPOS QUE FALTAM                          ║
# ║  Só os capítulos listados acima. Cada um se paga uma vez só.      ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess
from srt_utils import alinhar_versiculos
from models import Legenda
from whisper_utils import carregar_modelo_whisper

def duracao_ms(caminho):
    saida = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", str(caminho)], capture_output=True, text=True).stdout
    return int(float(saida.strip()) * 1000)

if not falta_tempo:
    print("✅ Nada a calcular — todos os capítulos já estão no cache.")
else:
    modelo = carregar_modelo_whisper(MODELO_WHISPER)

    for nome_cap, _ in falta_tempo:
        livro, capitulo = None, None
        for l, c, _ in ITENS:
            if l.nome_projeto(c) == nome_cap:
                livro, capitulo = l, c
                break

        audio = PASTA_AUDIO / f"{nome_cap}.mp3"
        print(f"\n🎙️  {nome_cap} — transcrevendo...")
        resultado = modelo.transcribe(str(audio), language=IDIOMA_AUDIO)

        legendas = []
        for seg in resultado.get("segments", []):
            texto = str(seg.get("text", "")).strip()
            if not texto:
                continue
            legendas.append(Legenda(
                id=len(legendas) + 1,
                inicio_ms=int(round(seg["start"] * 1000)),
                fim_ms=int(round(seg["end"] * 1000)),
                texto=texto,
            ))
        if not legendas:
            raise SystemExit(f"❌ {nome_cap}: Whisper não devolveu nada.")

        # O texto de referência é o que dá o número do versículo: o Whisper
        # não faz ideia de onde um versículo começa, o alinhamento é que casa
        # a sequência de palavras dos dois e empresta o tempo.
        texto_ref = bt.gerar_roteiro(versiculos_do_capitulo(livro, capitulo))
        tempos = alinhar_versiculos(texto_ref, legendas)
        if not tempos:
            raise SystemExit(f"❌ {nome_cap}: alinhamento não achou versículo nenhum.")

        esperados = len(versiculos_do_capitulo(livro, capitulo))
        if len(tempos) < esperados:
            print(f"   ⚠️  alinhou {len(tempos)} de {esperados} versículos")

        impressao = tc.impressao_de(audio, texto_ref, MODELO_WHISPER)
        tc.salvar(PASTA_TEMPOS, nome_cap, tempos, duracao_ms(audio), impressao)
        print(f"   ✅ {len(tempos)} versículos, {len(legendas)} blocos do Whisper")

    print(f"\n💾 cache agora tem {len(tc.capitulos_no_cache(PASTA_TEMPOS))} capítulo(s)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ✂️ 3/3 — CORTAR E CONCATENAR                                     ║
# ╚══════════════════════════════════════════════════════════════════╝

wav, srt, manifesto = compilar_selecao(
    ITENS,
    pasta_audio=PASTA_AUDIO,
    pasta_cache=PASTA_TEMPOS,
    texto_de=texto_de,
    pasta_saida=PASTA_SAIDA,
    nome_base=NOME,
)

print(f"🎵 {wav}")
print(f"📝 {srt}")
print(f"📋 {manifesto}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📊 RESUMO                                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

import json
dados = json.loads(Path(manifesto).read_text(encoding="utf-8"))

minutos, segundos = divmod(dados["duracao_s"], 60)
print(f"{dados['nome']}")
print(f"{dados['segmentos']} trechos · {int(minutos)}min {segundos:04.1f}s")
print("─" * 60)
for t in dados["trechos"]:
    print(f"  {t['inicio_compilado_s']:7.1f}s  {t['sigla']} {t['numero_capitulo']}:{t['versiculo']}")
print("─" * 60)
print()
print("O manifesto guarda, pra cada trecho, de qual versículo original ele veio.")
print("É por ele que a montagem de vídeo acha a mídia certa de cada pedaço.")
print()
print("Próximo passo: rodar o portao-qualidade.ipynb no resultado.")